# 六表质检与清洗V1
**基于模拟数据的业务分析项目。** 原始学习Notebook保留于01_data_quality.ipynb。本Notebook是整理后的复现入口。
输入六表raw，输出clean、quarantine和质量证据，不更改原始文件。

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'data/raw').exists():
    ROOT = ROOT.parent
assert (ROOT / 'data/raw/orders.csv').exists()
sys.path.insert(0, str(ROOT / 'scripts'))
pd.set_option('display.max_columns', 20)


## 运行确定性流水线
为什么：避免手动单元格执行顺序造成口径漂移。`run()`输入固定六表，执行标准化、去重、异常检查、SQL与分析输出；返回汇总字典。raw哈希不一致则停止。详细函数和处理规则见脚本及docs/methodology.md。

In [2]:
from run_project_v1 import run
summary = run()

{
  "data_type": "\u57fa\u4e8e\u6a21\u62df\u6570\u636e\u7684\u4e1a\u52a1\u5206\u6790\u9879\u76ee",
  "scope": "\u4e34\u65f6\u6709\u6548\u4e14\u6210\u672c\u5df2\u77e5\u8ba2\u5355\uff1b\u76f8\u540c\u8ba2\u5355\u7684\u6709\u6548\u6279\u51c6\u9000\u6b3e\u6309\u786e\u8ba4\u6708\uff1b\u5168\u90e8\u5df2\u8bb0\u5f55\u5e7f\u544a",
  "monthly": [
    {
      "month": "2026-08",
      "sales": 167384.39,
      "goods_cost": 67851.0,
      "gross_profit": 99533.39,
      "shipping_cost": 17100.6,
      "platform_cost": 16306.15,
      "before_refunds_ads": 66126.64,
      "units": 5946,
      "completed_orders": 5125,
      "refund_cost": 2528.02,
      "recorded_ad_cost": 12616.83,
      "contribution": 50981.79,
      "gross_margin_pct": 59.46396196
    },
    {
      "month": "2026-09",
      "sales": 135697.67,
      "goods_cost": 57774.0,
      "gross_profit": 77923.67,
      "shipping_cost": 14674.0,
      "platform_cost": 12927.04,
      "before_refunds_ads": 50322.63,
      "units": 5113,


## 粒度与重复复核
`read_csv()`输入派生统计文件并输出DataFrame。订单单SKU粒度、全字段重复去重；退款不能按order_id去重，广告不能按日期×SKU去重。

In [3]:
profiles = pd.read_csv(ROOT / "reports/tables/table_profiles.csv")
profiles

,table,grain,business_key,raw_rows,columns,exact_duplicate_excess,duplicate_key_rows,missing_cells
0,orders,一笔单SKU订单；根据数据字典及重复行全字段复核，非多商品订单明细,order_id,9970,12,18,36,12
1,products,一行一个商品SKU，成本两个月固定,sku,12,6,0,0,0
2,inventory,日期×SKU，共享库存日报,date+sku,732,6,0,0,0
3,refunds,一笔售后申请；order_id非主键，可有多笔退款,refund_id,312,7,0,0,0
4,ads,日期×平台×活动×SKU广告日报；conversions是订单数,date+platform+campaign_id+sku,2192,9,0,0,9
5,customers,一个注册客户；新老客标签相对8月1日固定,customer_id,4807,4,7,14,0


## 数据质量证据
`groupby().size()`对问题类型计数；一个记录可能有多项问题，因此问题条目不等于坏数据行数。只标记，不据此自动填补未知值。

In [4]:
issues = pd.read_csv(ROOT / "data/processed/quality_issues.csv")
issues.groupby(["table", "issue"]).size().rename("flagged_rows").reset_index()

,table,issue,flagged_rows
0,ads,missing:ad_spend,9
1,customers,exact_duplicate,7
2,inventory,inventory_order_difference,9
3,inventory,negative_inventory,3
4,inventory,stock_balance,3
5,inventory,stock_roll_forward,3
6,orders,country_format,22
7,orders,date_format:order_date,35
8,orders,exact_duplicate,18
9,orders,invalid_discount,3


## 检查清洗订单及隔离样本
输入clean订单，输出键、缺失与隔离记录数。clean含待核实记录，profit_orders才是临时利润样本。编号按文本读取。

In [5]:
orders = pd.read_csv(ROOT / "data/processed/orders_clean.csv", dtype={"order_id":str,"customer_id":str})
assert len(orders) == 9952 and orders.order_id.is_unique
print("客户编号缺失", orders.customer_id.isna().sum())
print("待核实订单", orders.needs_review.sum())
pd.read_csv(ROOT / "data/processed/orders_quarantine.csv")[["order_id","quantity","unit_price","order_status"]]

客户编号缺失 12
待核实订单 8


,order_id,quantity,unit_price,order_status
0,O0000074,999,32.00,completed
1,O0000091,0,39.78,completed
2,O0002524,0,36.90,completed
3,O0003613,1,-1.00,cancelled
4,O0004259,999,40.56,cancelled
5,O0004799,1,9999.00,completed
6,O0005861,1,9999.00,completed
7,O0008837,1,-1.00,completed


## 退款与广告待核实项
退款只退商品实收；超额退款不封顶。广告缺失费用不填0，疑似漏报网格需要业务确认。

In [6]:
display(pd.read_csv(ROOT / "data/processed/refunds_quarantine.csv")[["refund_id","order_id","refund_amount","order_sales"]])
display(pd.read_csv(ROOT / "data/processed/ads_missing_expected_rows.csv"))

,refund_id,order_id,refund_amount,order_sales
0,R000095,O0003304,645.75,86.11
1,R000176,O0006039,294.75,39.31


,date,platform,campaign_id,sku,basis
0,2026-08-04,Shopee,CAM-SHOPEE-SKU001,SKU001,观测固定活动每日模式；需确认是否停投或漏报
1,2026-09-24,Shopee,CAM-SHOPEE-SKU010,SKU010,观测固定活动每日模式；需确认是否停投或漏报
2,2026-08-22,TikTok Shop,CAM-TIKTOK-SKU004,SKU004,观测固定活动每日模式；需确认是否停投或漏报
3,2026-09-13,TikTok Shop,CAM-TIKTOK-SKU004,SKU004,观测固定活动每日模式；需确认是否停投或漏报


## 清洗成果与限制
六张clean CSV保留原始字段和新增标准字段；完整检查见reports/data_quality_report.md，未核实源系统错误仍保留。没有猜测修正异常价格、数量或未知采购成本。